In [1]:
from heapq import heappush, heappop

class TSPNode:
    def __init__(self, current_node, visited, g, h, path):
        self.current_node = current_node
        self.visited = visited # Set lưu trữ các đỉnh đã đi qua
        self.g = g             # Chi phí đường đi từ lúc xuất phát
        self.h = h             # Chi phí dự đoán đến đích
        self.f = g + h
        self.path = path       # Lưu mảng đường đi

    def __lt__(self, other):
        return self.f < other.f

class GraphTSP:
    def __init__(self, graph_matrix, nodes_list):
        self.graph = graph_matrix
        self.nodes = nodes_list
        self.num_nodes = len(nodes_list)

    # Heuristic: Lấy trọng số cạnh nhỏ nhất nối từ đỉnh hiện tại đến 1 đỉnh chưa thăm
    def heuristic(self, current_node_idx, visited):
        unvisited = [i for i in range(self.num_nodes) if i not in visited]
        if not unvisited:
            # Nếu đã thăm hết, chi phí dự đoán là đường quay về điểm xuất phát (đỉnh 0)
            return self.graph[current_node_idx][0]

        # Tìm cạnh có trọng số nhỏ nhất từ đỉnh hiện tại đến một đỉnh chưa thăm
        min_edge = float('inf')
        for v in unvisited:
            if self.graph[current_node_idx][v] < min_edge and self.graph[current_node_idx][v] != 0:
                min_edge = self.graph[current_node_idx][v]
        return min_edge if min_edge != float('inf') else 0

    def solve_a_star(self, start_idx=0):
        pq = []

        # Khởi tạo Node đầu tiên
        initial_visited = {start_idx}
        initial_h = self.heuristic(start_idx, initial_visited)

        start_node = TSPNode(start_idx, initial_visited, 0, initial_h, [self.nodes[start_idx]])
        heappush(pq, start_node)

        best_min_cost = float('inf')
        best_path = []

        while pq:
            current = heappop(pq)

            # Kiểm tra trạng thái đích: Đã thăm tất cả các đỉnh
            if len(current.visited) == self.num_nodes:
                # Cộng thêm chi phí quay lại đỉnh xuất phát
                return_cost = self.graph[current.current_node][start_idx]
                if return_cost > 0:
                    total_cost = current.g + return_cost
                    if total_cost < best_min_cost:
                        best_min_cost = total_cost
                        best_path = current.path + [self.nodes[start_idx]]
                continue

            # Duyệt các đỉnh kề chưa thăm
            for v in range(self.num_nodes):
                if v not in current.visited and self.graph[current.current_node][v] > 0:
                    cost_to_v = self.graph[current.current_node][v]

                    new_visited = current.visited.copy()
                    new_visited.add(v)
                    new_g = current.g + cost_to_v
                    new_h = self.heuristic(v, new_visited)
                    new_path = current.path + [self.nodes[v]]

                    child_node = TSPNode(v, new_visited, new_g, new_h, new_path)
                    heappush(pq, child_node)

        return best_path, best_min_cost

# --- CHẠY THỬ BÀI TOÁN ---
# Danh sách các đỉnh
cities = ['A', 'B', 'C', 'D']

# Ma trận trọng số (khoảng cách) giữa các đỉnh (0 là không có đường hoặc chính nó)
graph_matrix = [
    [0, 10, 15, 20],  # A đến A, B, C, D
    [10, 0, 35, 25],  # B đến A, B, C, D
    [15, 35, 0, 30],  # C đến A, B, C, D
    [20, 25, 30, 0]   # D đến A, B, C, D
]

tsp_solver = GraphTSP(graph_matrix, cities)
path, cost = tsp_solver.solve_a_star(start_idx=0) # Chọn A làm điểm xuất phát

print("=== Kết quả Bài toán Người giao hàng (TSP) bằng A* ===")
print("Lộ trình giao hàng ngắn nhất: ", " -> ".join(path))
print("Tổng chi phí (độ dài):", cost)

=== Kết quả Bài toán Người giao hàng (TSP) bằng A* ===
Lộ trình giao hàng ngắn nhất:  A -> B -> D -> C -> A
Tổng chi phí (độ dài): 80
